## Day 16 Completion Note

- The common CIC-IDS2018 training pipeline was recreated successfully.
- Training input contained 838,860 rows and 78 numeric features.
- Testing input contained 209,715 rows and 78 numeric features.
- The same 80/20 stratified split with `random_state=42` was used.
- Invalid numeric values were handled using training-data-derived medians.
- Target labels were encoded consistently for XGBoost and LightGBM.
- All five baseline models were re-trained using the common pipeline:
  - Decision Tree
  - Random Forest
  - AdaBoost
  - XGBoost
  - LightGBM
- Predictions were generated for all five models.
- Common evaluation metrics were calculated using weighted Precision, Recall, and F1 Score.
- Results were saved to:
  `results/day16_common_pipeline_results.csv`
- Saved results were successfully verified.

### Status
Day 16 – Re-run All 5 Models with Common Pipeline: **COMPLETED**

In [9]:
import os
import pandas as pd

print("Results file exists:", os.path.exists(results_path))

loaded_results = pd.read_csv(results_path)

print("\nSaved results:")
print(loaded_results.to_string(index=False))

Results file exists: True

Saved results:
        Model  Accuracy  Precision   Recall  F1 Score
Decision Tree  0.999952   0.999952 0.999952  0.999952
Random Forest  0.999952   0.999952 0.999952  0.999952
     AdaBoost  0.999943   0.999943 0.999943  0.999943
      XGBoost  0.999952   0.999952 0.999952  0.999952
     LightGBM  0.999952   0.999952 0.999952  0.999952


In [8]:
results_path = "../results/day16_common_pipeline_results.csv"

results_df.to_csv(results_path, index=False)

print("Day 16 results saved successfully.")
print("Results path:", results_path)

Day 16 results saved successfully.
Results path: ../results/day16_common_pipeline_results.csv


In [7]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

all_results = []

for name, y_pred in predictions.items():
    result = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        ),
        "F1 Score": f1_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )
    }

    all_results.append(result)

results_df = pd.DataFrame(all_results)

print("Common Pipeline Evaluation Results")
print("----------------------------------")
print(results_df.to_string(index=False))

Common Pipeline Evaluation Results
----------------------------------
        Model  Accuracy  Precision   Recall  F1 Score
Decision Tree  0.999952   0.999952 0.999952  0.999952
Random Forest  0.999952   0.999952 0.999952  0.999952
     AdaBoost  0.999943   0.999943 0.999943  0.999943
      XGBoost  0.999952   0.999952 0.999952  0.999952
     LightGBM  0.999952   0.999952 0.999952  0.999952


In [6]:
predictions = {}

for name, model in trained_models.items():
    print(f"Generating predictions: {name}")

    if name in ["XGBoost", "LightGBM"]:
        pred_encoded = model.predict(X_test_clean)
        predictions[name] = label_encoder.inverse_transform(
            pred_encoded.astype(int)
        )
    else:
        predictions[name] = model.predict(X_test_clean)

    print(f"{name}: {len(predictions[name])} predictions generated.")

print("\nAll 5 prediction sets generated successfully.")

Generating predictions: Decision Tree
Decision Tree: 209715 predictions generated.
Generating predictions: Random Forest
Random Forest: 209715 predictions generated.
Generating predictions: AdaBoost
AdaBoost: 209715 predictions generated.
Generating predictions: XGBoost
XGBoost: 209715 predictions generated.
Generating predictions: LightGBM
LightGBM: 209715 predictions generated.

All 5 prediction sets generated successfully.


In [5]:
import time

trained_models = {}
training_times = {}

for name, model in models.items():
    print(f"\n{'=' * 50}")
    print(f"Starting training: {name}")
    print(f"{'=' * 50}")

    start_time = time.time()

    if name in ["XGBoost", "LightGBM"]:
        model.fit(X_train_clean, y_train_encoded)
    else:
        model.fit(X_train_clean, y_train)

    elapsed_time = time.time() - start_time

    trained_models[name] = model
    training_times[name] = elapsed_time

    print(f"{name} training completed.")
    print(f"Training time: {elapsed_time:.2f} seconds")

print("\nAll 5 models trained successfully.")


Starting training: Decision Tree
Decision Tree training completed.
Training time: 4.00 seconds

Starting training: Random Forest
Random Forest training completed.
Training time: 25.20 seconds

Starting training: AdaBoost
AdaBoost training completed.
Training time: 164.03 seconds

Starting training: XGBoost
XGBoost training completed.
Training time: 9.82 seconds

Starting training: LightGBM
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.139670 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13834
[LightGBM] [Info] Number of data points in the train set: 838860, number of used features: 68
[LightGBM] [Info] Start training from score -0.451459
[LightGBM] [Info] Start training from score -1.690634
[LightGBM] [Info] Start training from score -1.720935
[LightGBM] [Warning] No further splits with positive gain, best gain: -in

In [4]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Label classes:")
print(label_encoder.classes_)

print("\nEncoded training labels:", y_train_encoded.shape)
print("Encoded testing labels :", y_test_encoded.shape)

print("\nEncoded values:", np.unique(y_train_encoded))

Label classes:
['Benign' 'FTP-BruteForce' 'SSH-Bruteforce']

Encoded training labels: (838860,)
Encoded testing labels : (209715,)

Encoded values: [0 1 2]


In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

models = {
    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        random_state=42,
        n_jobs=-1
    ),

    "AdaBoost": AdaBoostClassifier(
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        random_state=42,
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        n_jobs=-1
    ),

    "LightGBM": LGBMClassifier(
        random_state=42,
        objective="multiclass",
        num_class=3,
        n_jobs=-1
    )
}

print("Models created:")
for name, model in models.items():
    print(f"- {name}: {model.__class__.__name__}")

Models created:
- Decision Tree: DecisionTreeClassifier
- Random Forest: RandomForestClassifier
- AdaBoost: AdaBoostClassifier
- XGBoost: XGBClassifier
- LightGBM: LGBMClassifier


In [2]:
# Copy train/test inputs before cleaning
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

# Convert infinite values to NaN
X_train_clean = X_train_clean.replace([np.inf, -np.inf], np.nan)
X_test_clean = X_test_clean.replace([np.inf, -np.inf], np.nan)

# Calculate medians using training data only
train_medians = X_train_clean.median()

# Fill missing values using training medians
X_train_clean = X_train_clean.fillna(train_medians)
X_test_clean = X_test_clean.fillna(train_medians)

# Verify cleaned inputs
print("X_train_clean NaN:", X_train_clean.isna().sum().sum())
print("X_test_clean NaN :", X_test_clean.isna().sum().sum())

print("X_train_clean Inf:", np.isinf(X_train_clean).sum().sum())
print("X_test_clean Inf :", np.isinf(X_test_clean).sum().sum())

print("\nTraining input is finite:",
      np.isfinite(X_train_clean.to_numpy()).all())

print("Testing input is finite:",
      np.isfinite(X_test_clean.to_numpy()).all())

X_train_clean NaN: 0
X_test_clean NaN : 0
X_train_clean Inf: 0
X_test_clean Inf : 0

Training input is finite: True
Testing input is finite: True


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("../cic.csv")

X = df.drop(columns=["Label"])
y = df["Label"]

X_numeric = X.select_dtypes(include="number")

X_train, X_test, y_train, y_test = train_test_split(
    X_numeric,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features :", X_test.shape)
print("Training target  :", y_train.shape)
print("Testing target   :", y_test.shape)

Training features: (838860, 78)
Testing features : (209715, 78)
Training target  : (838860,)
Testing target   : (209715,)


# Day 16 – Re-run All 5 Models with Common Pipeline

## Objective
Re-run the five baseline ML models using the same common CIC-IDS2018 training pipeline.

## Models
1. Decision Tree
2. Random Forest
3. AdaBoost
4. XGBoost
5. LightGBM

## Common Training Input
- Dataset: CIC-IDS2018
- Training features: 78 numeric features
- Training rows: 838,860
- Testing rows: 209,715
- Train/test split: 80/20
- `random_state=42`
- Stratified target split
- Invalid numeric values handled using training-data-derived medians

## Objective of Day 16
- Recreate the common train/test input.
- Apply the same numeric cleaning process.
- Re-run all five baseline models.
- Generate predictions for each model.
- Calculate common evaluation metrics.
- Compare the resulting metrics across all five models.

No hyperparameter tuning will be performed on Day 16.